In [1]:
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import json
from datetime import datetime

In [2]:
COLUMN_NAMES = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes',
    'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
    'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
    'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
    'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
    'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
    'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
    'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty_level'
]



In [ ]:
# 2.2 FILE PATHS (UPDATE THESE FOR YOUR SYSTEM)
train_path = "../data/raw/KDDTrain+.txt"
test_path = "../data/raw/KDDTest+.txt"
output_dir = "../models"

In [4]:
train_df = pd.read_csv(train_path,header=None,names=COLUMN_NAMES)
test_df = pd.read_csv(test_path,header=None,names=COLUMN_NAMES)

In [5]:
train_df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty_level
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [6]:
test_df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty_level
0,0,tcp,private,REJ,0,0,0,0,0,0,...,0.04,0.06,0.00,0.00,0.0,0.0,1.00,1.00,neptune,21
1,0,tcp,private,REJ,0,0,0,0,0,0,...,0.00,0.06,0.00,0.00,0.0,0.0,1.00,1.00,neptune,21
2,2,tcp,ftp_data,SF,12983,0,0,0,0,0,...,0.61,0.04,0.61,0.02,0.0,0.0,0.00,0.00,normal,21
3,0,icmp,eco_i,SF,20,0,0,0,0,0,...,1.00,0.00,1.00,0.28,0.0,0.0,0.00,0.00,saint,15
4,1,tcp,telnet,RSTO,0,15,0,0,0,0,...,0.31,0.17,0.03,0.02,0.0,0.0,0.83,0.71,mscan,11


In [7]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125973 entries, 0 to 125972
Data columns (total 43 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   duration                     125973 non-null  int64  
 1   protocol_type                125973 non-null  object 
 2   service                      125973 non-null  object 
 3   flag                         125973 non-null  object 
 4   src_bytes                    125973 non-null  int64  
 5   dst_bytes                    125973 non-null  int64  
 6   land                         125973 non-null  int64  
 7   wrong_fragment               125973 non-null  int64  
 8   urgent                       125973 non-null  int64  
 9   hot                          125973 non-null  int64  
 10  num_failed_logins            125973 non-null  int64  
 11  logged_in                    125973 non-null  int64  
 12  num_compromised              125973 non-null  int64  
 13 

In [8]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22544 entries, 0 to 22543
Data columns (total 43 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   duration                     22544 non-null  int64  
 1   protocol_type                22544 non-null  object 
 2   service                      22544 non-null  object 
 3   flag                         22544 non-null  object 
 4   src_bytes                    22544 non-null  int64  
 5   dst_bytes                    22544 non-null  int64  
 6   land                         22544 non-null  int64  
 7   wrong_fragment               22544 non-null  int64  
 8   urgent                       22544 non-null  int64  
 9   hot                          22544 non-null  int64  
 10  num_failed_logins            22544 non-null  int64  
 11  logged_in                    22544 non-null  int64  
 12  num_compromised              22544 non-null  int64  
 13  root_shell      

In [9]:
train_df.describe()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty_level
count,125973.00000,1.259730e+05,1.259730e+05,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,...,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000,125973.000000
mean,287.14465,4.556674e+04,1.977911e+04,0.000198,0.022687,0.000111,0.204409,0.001222,0.395736,0.279250,...,115.653005,0.521242,0.082951,0.148379,0.032542,0.284452,0.278485,0.118832,0.120240,19.504060
std,2604.51531,5.870331e+06,4.021269e+06,0.014086,0.253530,0.014366,2.149968,0.045239,0.489010,23.942042,...,110.702741,0.448949,0.188922,0.308997,0.112564,0.444784,0.445669,0.306557,0.319459,2.291503
min,0.00000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.00000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,10.000000,0.050000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,18.000000
50%,0.00000,4.400000e+01,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,63.000000,0.510000,0.020000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,20.000000
75%,0.00000,2.760000e+02,5.160000e+02,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,...,255.000000,1.000000,0.070000,0.060000,0.020000,1.000000,1.000000,0.000000,0.000000,21.000000
max,42908.00000,1.379964e+09,1.309937e+09,1.000000,3.000000,3.000000,77.000000,5.000000,1.000000,7479.000000,...,255.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,21.000000


In [10]:
test_df.describe()

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,difficulty_level
count,22544.000000,2.254400e+04,2.254400e+04,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,...,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000,22544.000000
mean,218.859076,1.039545e+04,2.056019e+03,0.000311,0.008428,0.000710,0.105394,0.021647,0.442202,0.119899,...,140.750532,0.608722,0.090540,0.132261,0.019638,0.097814,0.099426,0.233385,0.226683,18.017965
std,1407.176612,4.727864e+05,2.121930e+04,0.017619,0.142599,0.036473,0.928428,0.150328,0.496659,7.269597,...,111.783972,0.435688,0.220717,0.306268,0.085394,0.273139,0.281866,0.387229,0.400875,4.270361
min,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,15.000000,0.070000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,17.000000
50%,0.000000,5.400000e+01,4.600000e+01,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,168.000000,0.920000,0.010000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,20.000000
75%,0.000000,2.870000e+02,6.010000e+02,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,...,255.000000,1.000000,0.060000,0.030000,0.010000,0.000000,0.000000,0.360000,0.170000,21.000000
max,57715.000000,6.282565e+07,1.345927e+06,1.000000,3.000000,3.000000,101.000000,4.000000,1.000000,796.000000,...,255.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,21.000000


In [11]:
train_df.isna().sum()

duration                       0
protocol_type                  0
service                        0
flag                           0
src_bytes                      0
dst_bytes                      0
land                           0
wrong_fragment                 0
urgent                         0
hot                            0
num_failed_logins              0
logged_in                      0
num_compromised                0
root_shell                     0
su_attempted                   0
num_root                       0
num_file_creations             0
num_shells                     0
num_access_files               0
num_outbound_cmds              0
is_host_login                  0
is_guest_login                 0
count                          0
srv_count                      0
serror_rate                    0
srv_serror_rate                0
rerror_rate                    0
srv_rerror_rate                0
same_srv_rate                  0
diff_srv_rate                  0
srv_diff_h

In [12]:
test_df.isnull().sum()

duration                       0
protocol_type                  0
service                        0
flag                           0
src_bytes                      0
dst_bytes                      0
land                           0
wrong_fragment                 0
urgent                         0
hot                            0
num_failed_logins              0
logged_in                      0
num_compromised                0
root_shell                     0
su_attempted                   0
num_root                       0
num_file_creations             0
num_shells                     0
num_access_files               0
num_outbound_cmds              0
is_host_login                  0
is_guest_login                 0
count                          0
srv_count                      0
serror_rate                    0
srv_serror_rate                0
rerror_rate                    0
srv_rerror_rate                0
same_srv_rate                  0
diff_srv_rate                  0
srv_diff_h

In [13]:
train_df.isnull().sum()

duration                       0
protocol_type                  0
service                        0
flag                           0
src_bytes                      0
dst_bytes                      0
land                           0
wrong_fragment                 0
urgent                         0
hot                            0
num_failed_logins              0
logged_in                      0
num_compromised                0
root_shell                     0
su_attempted                   0
num_root                       0
num_file_creations             0
num_shells                     0
num_access_files               0
num_outbound_cmds              0
is_host_login                  0
is_guest_login                 0
count                          0
srv_count                      0
serror_rate                    0
srv_serror_rate                0
rerror_rate                    0
srv_rerror_rate                0
same_srv_rate                  0
diff_srv_rate                  0
srv_diff_h

In [14]:
train_df.duplicated().sum()

np.int64(0)

In [15]:
print(train_df.shape)
print(test_df.shape)

(125973, 43)
(22544, 43)


In [17]:
print(train_df.dtypes.value_counts())

int64      24
float64    15
object      4
Name: count, dtype: int64


In [18]:
train_df['label'].value_counts()

label
normal             67343
neptune            41214
satan               3633
ipsweep             3599
portsweep           2931
smurf               2646
nmap                1493
back                 956
teardrop             892
warezclient          890
pod                  201
guess_passwd          53
buffer_overflow       30
warezmaster           20
land                  18
imap                  11
rootkit               10
loadmodule             9
ftp_write              8
multihop               7
phf                    4
perl                   3
spy                    2
Name: count, dtype: int64

In [19]:
test_df['label'].value_counts()

label
normal             9711
neptune            4657
guess_passwd       1231
mscan               996
warezmaster         944
apache2             737
satan               735
processtable        685
smurf               665
back                359
snmpguess           331
saint               319
mailbomb            293
snmpgetattack       178
portsweep           157
ipsweep             141
httptunnel          133
nmap                 73
pod                  41
buffer_overflow      20
multihop             18
named                17
ps                   15
sendmail             14
xterm                13
rootkit              13
teardrop             12
xlock                 9
land                  7
xsnoop                4
ftp_write             3
loadmodule            2
worm                  2
perl                  2
sqlattack             2
udpstorm              2
phf                   2
imap                  1
Name: count, dtype: int64

In [20]:
print(set(train_df['label'].unique()))
print(set(test_df['label'].unique()))

{'warezclient', 'normal', 'back', 'warezmaster', 'guess_passwd', 'ipsweep', 'teardrop', 'ftp_write', 'land', 'pod', 'perl', 'buffer_overflow', 'nmap', 'multihop', 'imap', 'neptune', 'loadmodule', 'spy', 'satan', 'portsweep', 'smurf', 'rootkit', 'phf'}
{'saint', 'xterm', 'udpstorm', 'normal', 'back', 'snmpguess', 'warezmaster', 'guess_passwd', 'ipsweep', 'named', 'processtable', 'snmpgetattack', 'teardrop', 'ftp_write', 'sqlattack', 'land', 'pod', 'perl', 'xsnoop', 'buffer_overflow', 'nmap', 'ps', 'worm', 'multihop', 'apache2', 'imap', 'sendmail', 'neptune', 'xlock', 'loadmodule', 'satan', 'mailbomb', 'portsweep', 'smurf', 'httptunnel', 'rootkit', 'mscan', 'phf'}


## MAP ATTACK CATEGORY

In [ ]:
# define mapping once
ATTACK_MAPPING = {
    'normal': 'normal',
    # DoS
    'neptune': 'dos', 'smurf': 'dos', 'back': 'dos',
    'teardrop': 'dos', 'pod': 'dos', 'land': 'dos',
    'apache2': 'dos', 'udpstorm': 'dos', 'processtable': 'dos',
    'worm': 'dos', 'mailbomb': 'dos',
    # Probe
    'portsweep': 'probe', 'ipsweep': 'probe',
    'nmap': 'probe', 'satan': 'probe',
    'mscan': 'probe', 'saint': 'probe',
    # R2L
    'guess_passwd': 'r2l', 'warezmaster': 'r2l',
    'imap': 'r2l', 'multihop': 'r2l', 'phf': 'r2l',
    'spy': 'r2l', 'warezclient': 'r2l',
    'ftp_write': 'r2l', 'named': 'r2l',
    'sendmail': 'r2l', 'snmpgetattack': 'r2l',
    'snmpguess': 'r2l', 'xlock': 'r2l', 'xsnoop': 'r2l',
    # U2R
    'buffer_overflow': 'u2r', 'loadmodule': 'u2r',
    'rootkit': 'u2r', 'perl': 'u2r',
    'sqlattack': 'u2r', 'xterm': 'u2r',
    'httptunnel': 'u2r', 'ps': 'u2r'
}


# apply mapping
train_df['is_attack_multiclass'] = train_df['label'].map(ATTACK_MAPPING).fillna('unknown')
test_df['is_attack_multiclass'] = test_df['label'].map(ATTACK_MAPPING).fillna('unknown')

In [22]:
train_df['is_attack_binary'] = (train_df['label'] != 'normal').astype(int)
test_df['is_attack_binary'] = (test_df['label'] != 'normal').astype(int)

In [23]:
train_df[['label','is_attack_multiclass','is_attack_binary']].head()

,label,is_attack_multiclass,is_attack_binary
0,normal,normal,0
1,normal,normal,0
2,neptune,dos,1
3,normal,normal,0
4,normal,normal,0


In [24]:
test_df[['label','is_attack_multiclass','is_attack_binary']]

,label,is_attack_multiclass,is_attack_binary
0,neptune,dos,1
1,neptune,dos,1
2,normal,normal,0
3,saint,probe,1
4,mscan,probe,1
...,...,...,...
22539,normal,normal,0
22540,normal,normal,0
22541,back,dos,1
22542,normal,normal,0


In [25]:
train_df['duration'].value_counts()

duration
0        115955
1          1989
2           843
3           557
4           351
          ...  
12039         1
9235          1
41254         1
36613         1
679           1
Name: count, Length: 2981, dtype: int64

In [56]:
train_df['is_attack_binary'].value_counts()

is_attack_binary
0    67343
1    58630
Name: count, dtype: int64

In [ ]:
def sanitize_network_data(df):

    df = df.copy()

    # 1. No negative values
    for col in ['duration', 'src_bytes', 'dst_bytes']:
        if col in df.columns:
            df[col] = df[col].clip(lower=0)

    # 2. Rates must be between 0 and 1
    for col in df.columns:
        if col.endswith('_rate'):
            df[col] = df[col].clip(0, 1)

    # 3. Binary columns → 0 / 1
    binary_cols = [
        'land', 'logged_in', 'root_shell', 'su_attempted',
        'is_host_login', 'is_guest_login'
    ]
    for col in binary_cols:
        if col in df.columns:
            df[col] = (df[col] > 0).astype(int)

    return df
train_df = sanitize_network_data(train_df)
test_df  = sanitize_network_data(test_df)

In [61]:
train_df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty_level,is_attack_multiclass,is_attack_binary
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.00,0.00,0.00,0.05,0.00,normal,20,normal,0
1,0,udp,other,SF,146,0,0,0,0,0,...,0.88,0.00,0.00,0.00,0.00,0.00,normal,15,normal,0
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19,dos,1
3,0,tcp,http,SF,232,8153,0,0,0,0,...,0.03,0.04,0.03,0.01,0.00,0.01,normal,21,normal,0
4,0,tcp,http,SF,199,420,0,0,0,0,...,0.00,0.00,0.00,0.00,0.00,0.00,normal,21,normal,0


In [29]:
train_df['duration'].value_counts()

duration
0        115955
1          1989
2           843
3           557
4           351
          ...  
12039         1
9235          1
41254         1
36613         1
679           1
Name: count, Length: 2981, dtype: int64

In [30]:
def fit_iqr_bounds(df):
    bounds = {}
    numeric_cols = df.select_dtypes(include='number').columns

    for col in numeric_cols:
        if df[col].nunique() > 10:  # Only for continuous variables, skip binary and encoded columns
            Q1 = df[col].quantile(0.01)  # Use 1% and 99% for more aggressive clipping
            Q3 = df[col].quantile(0.99)
            IQR = Q3 - Q1
            lower = Q1 - 3 * IQR
            upper = Q3 + 3 * IQR
            bounds[col] = (lower, upper)

    return bounds


In [31]:
def apply_iqr_bounds(df, bounds):
    df = df.copy()
    for col, (lower, upper) in bounds.items():
        if col in df.columns:
            df[col] = df[col].clip(lower, upper)
    return df


In [32]:
iqr_bounds = fit_iqr_bounds(train_df)

train_df = apply_iqr_bounds(train_df, iqr_bounds)
test_df  = apply_iqr_bounds(test_df, iqr_bounds)

In [33]:
train_df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty_level,is_attack_multiclass,is_attack_binary
0,0.0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.00,0.00,0.00,0.05,0.00,normal,20,normal,0
1,0.0,udp,other,SF,146,0,0,0,0,0,...,0.88,0.00,0.00,0.00,0.00,0.00,normal,15,normal,0
2,0.0,tcp,private,S0,0,0,0,0,0,0,...,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19,dos,1
3,0.0,tcp,http,SF,232,8153,0,0,0,0,...,0.03,0.04,0.03,0.01,0.00,0.01,normal,21,normal,0
4,0.0,tcp,http,SF,199,420,0,0,0,0,...,0.00,0.00,0.00,0.00,0.00,0.00,normal,21,normal,0


In [34]:
train_df['duration'].value_counts()

duration
0.0        115955
1.0          1989
2.0           843
3.0           557
4.0           351
            ...  
731.0           1
31041.0         1
6139.0          1
11750.0         1
703.0           1
Name: count, Length: 2784, dtype: int64

#### Connection Intensity Ratio

### Connection Intensity Ratio

**What it means:**

* Measures how focused a source’s connections are on a single service of a host.
* **High (≈1):** Most connections target one main service → normal
* **Low (≫0):** Connections spread across many services → suspicious

**Why it matters:**

* Detects focused vs. dispersed traffic
* Helps IDS spot reconnaissance, port scanning, or probing
* Differentiates legitimate users from attackers

**Analogy:**

* High → everyone goes straight to the main department → normal
* Low → visitors check every department → suspicious

In [35]:
fe_df = train_df.copy()

In [36]:
# Connection Intensity
fe_df['connection_intensity_ratio'] = fe_df['count'] / (fe_df['srv_count'] + 1)

### Total Bytes (Discretized)

**What it means:**

* Measures total data transferred (sent + received) and groups it into 5 categories: **very_low, low, medium, high, very_high**.
* **very_low / low:** Tiny transfers → may indicate scans or probes
* **medium:** Normal traffic
* **high / very_high:** Large transfers → may indicate bulk data or exfiltration

**Why it matters:**

* Reduces skew from extreme values
* Captures intensity of activity without being dominated by outliers
* Helps IDS spot unusual or suspicious data transfers

**Analogy:**

* very_low / low → brief visits → possibly probing
* medium → normal usage
* high / very_high → carrying large stacks → potential attack or sensitive operations

In [37]:
fe_df['total_bytes'] = fe_df['src_bytes'] + fe_df['dst_bytes']

### Byte Ratio

**What it means:**

* Measures traffic imbalance between source and destination.
* **≈1:** Balanced → normal exchange
* **≫1:** Source sends much more → potential exfiltration
* **≪1:** Destination sends much more → unusual flow

**Why it matters:**

* Detects asymmetric traffic, a common sign of malicious activity
* Flags data exfiltration or bot/C2 communication
* Scale-independent → works across small and large connections

**Analogy:**

* Balanced → fair exchange → normal
* Extreme → one-sided transfer → suspicious

In [38]:
fe_df['byte_ratio'] = fe_df['src_bytes'] / (fe_df['dst_bytes'] + 1)

### Combined Error Rate

**What it means:**

* Summarizes connection reliability by averaging several error indicators.
* **Low (~0):** Connections mostly successful → normal behavior
* **High (~1):** Many failed or rejected attempts → may indicate scanning or DoS

**Why it matters:**

* Captures instability or failure patterns → strong signal of suspicious activity
* Helps IDS detect port scans, probing, or DoS attacks

**Analogy:**

* Low → smooth, successful entry → normal
* High → repeated failed attempts → suspicious

In [39]:
# --- Error Rates ---
fe_df['combined_error_rate'] = fe_df[['serror_rate','srv_serror_rate','rerror_rate','srv_rerror_rate']].mean(axis=1)

### Service Diversity Score

**What it means:**

* Measures how evenly a host’s connections are spread across services.
* **High (~1):** Traffic concentrated on few services → normal usage
* **Low (~0):** Traffic spread across many services → may indicate scanning

**Why it matters:**

* Detects whether traffic is focused or dispersed per host
* Helps identify legitimate users vs. attackers performing reconnaissance

**Analogy:**

* High → visitors stick to main departments → normal
* Low → visitors check many departments → suspicious

In [40]:
fe_df['service_diversity_score'] = 1 - fe_df[['same_srv_rate','dst_host_same_srv_rate']].mean(axis=1)

### Host Diversity

**What it means:**

* Measures how spread out connections are across services on a host.
* **High (~1):** Connections target many services → may indicate scanning
* **Low (~0):** Most traffic goes to one service → normal behavior

**Why it matters:**

* Detects probing or reconnaissance activity
* Complements other signals like host concentration

**Analogy:**

* Low → visitor focuses on one area → normal
* High → visitor checks every department → suspicious

In [41]:
fe_df['host_diversity'] = fe_df['dst_host_diff_srv_rate'] * fe_df['dst_host_srv_diff_host_rate']

### Traffic Burst

**What it means:**

* Detects sudden spikes in network activity.
* **True (1):** Many connections in a very short time → suspicious
* **False (0):** Normal traffic patterns

**Why it matters:**

* Flags DoS attacks, floods, or rapid scanning
* Normal users rarely generate high-frequency bursts

**Analogy:**

* False → steady flow of visitors → normal
* True → crowd rushes in at once → potential attack

In [42]:
fe_df['traffic_burst'] = ((fe_df['count'] > fe_df['count'].quantile(0.75)) & 
                          (fe_df['duration'] < fe_df['duration'].quantile(0.25))).astype(int)

### Privilege Indicators

**What it means:**

* Measures attempts to gain elevated or unauthorized access.
* **privilege_indicators > 0** → potential privilege escalation
* **has_privilege_attempt = 1** → any attempt at elevated access → suspicious
* **0** → normal activity

**Why it matters:**

* Privilege escalation is rare in normal traffic but common in intrusions
* Helps IDS detect malicious intent, not just traffic volume

**Analogy:**

* 0 → visitor sticks to normal areas → normal
* > 0 → visitor tries restricted areas → suspicious/attacker

In [43]:
priv_cols = ['root_shell','su_attempted','num_root','num_file_creations','num_shells','num_access_files']
fe_df['privilege_attempt'] = (fe_df[priv_cols].sum(axis=1) > 0).astype(int)

In [44]:
def add_network_features(df, burst_count_q=None, burst_duration_q=None):
    df = df.copy()

    # Connection Intensity
    df['connection_intensity_ratio'] = df['count'] / (df['srv_count'] + 1)

    # Traffic Volume
    df['total_bytes'] = df['src_bytes'] + df['dst_bytes']
    df['byte_ratio'] = df['src_bytes'] / (df['dst_bytes'] + 1)

    # Error Rates
    error_cols = [
        'serror_rate', 'srv_serror_rate',
        'rerror_rate', 'srv_rerror_rate'
    ]
    df['combined_error_rate'] = df[error_cols].mean(axis=1)

    # Diversity Features
    df['service_diversity_score'] = 1 - df[
        ['same_srv_rate', 'dst_host_same_srv_rate']
    ].mean(axis=1)

    df['host_diversity'] = (
        df['dst_host_diff_srv_rate'] *
        df['dst_host_srv_diff_host_rate']
    )

    # Traffic Burst (NO leakage)
    if burst_count_q is None:
        burst_count_q = df['count'].quantile(0.75)
    if burst_duration_q is None:
        burst_duration_q = df['duration'].quantile(0.25)

    df['traffic_burst'] = (
        (df['count'] > burst_count_q) &
        (df['duration'] < burst_duration_q)
    ).astype(int)

    # Privilege Escalation
    priv_cols = [
        'root_shell', 'su_attempted', 'num_root',
        'num_file_creations', 'num_shells', 'num_access_files'
    ]
    df['privilege_attempt'] = (df[priv_cols].sum(axis=1) > 0).astype(int)

    return df, burst_count_q, burst_duration_q


In [45]:
train_df, count_q, duration_q = add_network_features(train_df)

In [46]:
test_df, _, _ = add_network_features(
    test_df,
    burst_count_q=count_q,
    burst_duration_q=duration_q
)


In [47]:
def add_security_engineered_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1. Login anomaly score (adapted)
    df['login_anomaly_score'] = df['num_compromised'] / (df['duration'] + 1)

    # 2. Privileged escalation ratio
    df['privileged_ratio'] = df['root_shell'] / (df['hot'] + 1)

    # 3. Service rarity (global frequency-based)
    service_freq = df['service'].value_counts(normalize=True)
    df['service_rarity'] = df['service'].map(
        lambda x: 1.0 / service_freq.get(x, 1e-6)
    )

    # 4. Command density
    df['command_density'] = df['hot'] / (df['src_bytes'] + 1)

    # 5. Inbound / outbound byte ratio
    df['inbound_outbound_ratio'] = df['dst_bytes'] / (df['src_bytes'] + 1)

    # 6. Safety cleanup
    df.replace([np.inf, -np.inf], 0, inplace=True)
    df.fillna(0, inplace=True)

    return df


In [48]:
train_df = add_security_engineered_features(train_df)
test_df  = add_security_engineered_features(test_df)

In [49]:
train_df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,combined_error_rate,service_diversity_score,host_diversity,traffic_burst,privilege_attempt,login_anomaly_score,privileged_ratio,service_rarity,command_density,inbound_outbound_ratio
0,0.0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.0,0.415,0.0,0,0,0.0,0.0,18.363411,0.0,0.000000
1,0.0,udp,other,SF,146,0,0,0,0,0,...,0.0,0.960,0.0,0,0,0.0,0.0,28.899518,0.0,0.000000
2,0.0,tcp,private,S0,0,0,0,0,0,0,...,0.5,0.925,0.0,0,0,0.0,0.0,5.764563,0.0,0.000000
3,0.0,tcp,http,SF,232,8153,0,0,0,0,...,0.1,0.000,0.0,0,0,0.0,0.0,3.122936,0.0,34.991416
4,0.0,tcp,http,SF,199,420,0,0,0,0,...,0.0,0.000,0.0,0,0,0.0,0.0,3.122936,0.0,2.100000


In [50]:
test_df.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,combined_error_rate,service_diversity_score,host_diversity,traffic_burst,privilege_attempt,login_anomaly_score,privileged_ratio,service_rarity,command_density,inbound_outbound_ratio
0,0.0,tcp,private,REJ,0,0,0,0,0,0,...,0.500,0.960,0.0000,0,0,0.0,0.0,4.722245,0.0,0.0
1,0.0,tcp,private,REJ,0,0,0,0,0,0,...,0.500,0.995,0.0000,0,0,0.0,0.0,4.722245,0.0,0.0
2,2.0,tcp,ftp_data,SF,12983,0,0,0,0,0,...,0.000,0.195,0.0008,0,0,0.0,0.0,26.491187,0.0,0.0
3,0.0,icmp,eco_i,SF,20,0,0,0,0,0,...,0.000,0.000,0.0000,0,0,0.0,0.0,86.045802,0.0,0.0
4,1.0,tcp,telnet,RSTO,0,15,0,0,0,0,...,0.405,0.345,0.0034,0,0,0.0,0.0,13.864699,0.0,15.0


In [51]:
train_df.columns

Index(['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
       'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
       'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
       'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
       'num_access_files', 'num_outbound_cmds', 'is_host_login',
       'is_guest_login', 'count', 'srv_count', 'serror_rate',
       'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
       'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
       'dst_host_srv_count', 'dst_host_same_srv_rate',
       'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
       'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
       'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
       'dst_host_srv_rerror_rate', 'label', 'difficulty_level',
       'is_attack_multiclass', 'is_attack_binary',
       'connection_intensity_ratio', 'total_bytes', 'byte_ratio',
       'combined_error_rate', 'serv

In [52]:
test_df.columns

Index(['duration', 'protocol_type', 'service', 'flag', 'src_bytes',
       'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
       'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
       'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
       'num_access_files', 'num_outbound_cmds', 'is_host_login',
       'is_guest_login', 'count', 'srv_count', 'serror_rate',
       'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
       'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
       'dst_host_srv_count', 'dst_host_same_srv_rate',
       'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
       'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
       'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
       'dst_host_srv_rerror_rate', 'label', 'difficulty_level',
       'is_attack_multiclass', 'is_attack_binary',
       'connection_intensity_ratio', 'total_bytes', 'byte_ratio',
       'combined_error_rate', 'serv

In [53]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125973 entries, 0 to 125972
Data columns (total 58 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   duration                     125973 non-null  float64
 1   protocol_type                125973 non-null  object 
 2   service                      125973 non-null  object 
 3   flag                         125973 non-null  object 
 4   src_bytes                    125973 non-null  int64  
 5   dst_bytes                    125973 non-null  int64  
 6   land                         125973 non-null  int64  
 7   wrong_fragment               125973 non-null  int64  
 8   urgent                       125973 non-null  int64  
 9   hot                          125973 non-null  int64  
 10  num_failed_logins            125973 non-null  int64  
 11  logged_in                    125973 non-null  int64  
 12  num_compromised              125973 non-null  int64  
 13 

In [54]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22544 entries, 0 to 22543
Data columns (total 58 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   duration                     22544 non-null  float64
 1   protocol_type                22544 non-null  object 
 2   service                      22544 non-null  object 
 3   flag                         22544 non-null  object 
 4   src_bytes                    22544 non-null  int64  
 5   dst_bytes                    22544 non-null  int64  
 6   land                         22544 non-null  int64  
 7   wrong_fragment               22544 non-null  int64  
 8   urgent                       22544 non-null  int64  
 9   hot                          22544 non-null  int64  
 10  num_failed_logins            22544 non-null  int64  
 11  logged_in                    22544 non-null  int64  
 12  num_compromised              22544 non-null  int64  
 13  root_shell      

In [55]:
# Convert string/object columns to category
categorical_cols = ['protocol_type', 'service', 'flag']
for col in categorical_cols:
    train_df[col] = train_df[col].astype('category')

train_df['is_attack_multiclass'] = train_df['is_attack_multiclass'].astype('category')

In [56]:
# Convert string/object columns to category
categorical_cols = ['protocol_type', 'service', 'flag']
for col in categorical_cols:
    test_df[col] = test_df[col].astype('category')

test_df['is_attack_multiclass'] = test_df['is_attack_multiclass'].astype('category')

In [57]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 125973 entries, 0 to 125972
Data columns (total 58 columns):
 #   Column                       Non-Null Count   Dtype   
---  ------                       --------------   -----   
 0   duration                     125973 non-null  float64 
 1   protocol_type                125973 non-null  category
 2   service                      125973 non-null  category
 3   flag                         125973 non-null  category
 4   src_bytes                    125973 non-null  int64   
 5   dst_bytes                    125973 non-null  int64   
 6   land                         125973 non-null  int64   
 7   wrong_fragment               125973 non-null  int64   
 8   urgent                       125973 non-null  int64   
 9   hot                          125973 non-null  int64   
 10  num_failed_logins            125973 non-null  int64   
 11  logged_in                    125973 non-null  int64   
 12  num_compromised              125973 non-null

In [58]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22544 entries, 0 to 22543
Data columns (total 58 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   duration                     22544 non-null  float64 
 1   protocol_type                22544 non-null  category
 2   service                      22544 non-null  category
 3   flag                         22544 non-null  category
 4   src_bytes                    22544 non-null  int64   
 5   dst_bytes                    22544 non-null  int64   
 6   land                         22544 non-null  int64   
 7   wrong_fragment               22544 non-null  int64   
 8   urgent                       22544 non-null  int64   
 9   hot                          22544 non-null  int64   
 10  num_failed_logins            22544 non-null  int64   
 11  logged_in                    22544 non-null  int64   
 12  num_compromised              22544 non-null  int64   
 13  r

In [59]:
print(train_df.shape)
print(test_df.shape)

(125973, 58)
(22544, 58)


In [60]:
test_df.shape

(22544, 58)

In [61]:
# Save cleaned train
train_df.to_csv('../data/processed/cleaned_network_intrusion_detection_train.csv', index=False)

# Save cleaned test
test_df.to_csv('../data/processed/cleaned_network_intrusion_detection_test.csv', index=False)
